In [ ]:
# Setup: code comes from GitHub onto the VM disk; data + checkpoints live on Drive (see src/config.py)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/s1156052/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN (never write it in the notebook)
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!apt-get -qq install -y cd-hit > /dev/null


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/s1156052/NeoAntigen


In [ ]:
# Collect once (dated snapshot on Drive): every CEDAR neoantigen with a disease_group, plus viral/TAA background negatives
import pandas as pd
from src import config, pancancer
pd.set_option("display.width", 200)

raw = pancancer.get_raw(config.DATA_DIR, refresh=False)
print(raw.groupby("group").size())


using snapshot pancancer_cedar_raw_20261110.csv
group
neo      25011
taa      18161
viral    30488
dtype: int64


In [ ]:
# How much data each disease has (decides which diseases can be their own train/test arm)
pancancer.summary(raw)


,disease_group,rows,unique_seq,positive_seq,exp_negative_seq,unknown_seq,median_year
0,melanoma,7106,7102,1534,5542,26,2021
1,lung,2453,2453,644,1792,17,2019
2,colorectal,8966,8957,462,8466,30,2019
3,pancreatic,1005,1005,235,768,2,2021
4,breast,483,483,220,250,13,2022
5,other,453,453,186,242,25,2024
6,bladder,183,183,172,11,0,2022
7,renal,242,242,142,100,0,2025
8,hepatocellular,560,560,138,420,2,2021
9,brain,527,527,133,393,1,2021


In [ ]:
# Disease names that fell into "other": check none of them belongs to a named group or is not a cancer
pancancer.discover_terms(raw).head(40)


In [ ]:
# Step 2: frozen dataset = one manifest (unique peptides, disease_group, CD-HIT cluster, outer fold). Needs cd-hit (Colab).
# neg_mode "exp": disease-matched T cell-negative neoantigens (primary) | "bg": viral/TAA background (the paper's choice)
from src import pancancer_split as ps
NAME, NEG_MODE = "pancancer_cedar_v1", "exp"
manifest, summary = ps.make_dataset(raw, config.DATA_DIR, config.WORK_DIR, name=NAME, neg_mode=NEG_MODE)
summary


In [ ]:
# One arm (train diseases) x one outer test fold -> the dataframe the runner expects; written to DATA_DIR/<NAME>/<arm>_f<k>/
ARM, FOLD = "pooled", 0
df = ps.arm_frame(manifest, ARM, FOLD, neg_ratio=5)
ps.write_arm(df, config.DATA_DIR, NAME)
ps.eval_table(df)     # N+/N- per disease on test: all rows vs rows not similar to training (reportable needs >= 50 each)


In [ ]:
# Every arm x fold (cheap; each frame is checked for sequence/cluster leakage)
for arm in ps.ARMS:
    for k in range(5):
        ps.write_arm(ps.arm_frame(manifest, arm, k), config.DATA_DIR, NAME)


In [ ]:
# Step 3 (paper-style): raw -> pos/neg (experimental negatives) -> CD-HIT 0.4 once -> length <= 35 -> 80:20
# colorectal / lung / melanoma each 80:20; pooled = the three together; other cancers = extra test only
from src import pancancer_dataset as pds
NAME_P = "pancancer_paper_v1"
res = pds.make_paper_dataset(raw, config.DATA_DIR, config.WORK_DIR, name=NAME_P)
print(res["table1"].to_string(), "\n")      # rows per stage and group: how much CD-HIT removes
print(res["table2"].to_string(), "\n")      # train / test sizes per set
print(res["similarity"].to_string(), "\n")  # max identity of each held-out set to the pooled train (violations should be 0)
print("limitations:", res["limitations"])


In [ ]:
# Load one set for the runner: data.load_dataset(config.DATA_DIR / NAME_P / "colorectal" | "lung" | "melanoma" | "pooled")
from src import data
df_pool = data.load_dataset(config.DATA_DIR / NAME_P / "pooled")
pds.eval_table(df_pool)     # test sets of the pooled model: 3 cancers + each other cancer + all other solid
